<a href="https://colab.research.google.com/github/LeJ-04/web-datamining-semantics/blob/main/notebooks/reasoning_kge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part C : Reasoning & Knowledge Graph Embeddings

This notebook implements the full reasoning and KGE pipeline for the Premier League Football Knowledge Graph. It is structured into four sections:

- C.1 : Prepare KGE datasets (filter, split, OOV prevention)
- C.2 : Symbolic reasoning with SWRL rules (infer new knowledge)
- C.3 :  Train two KGE models: TransE and ComplEx
- C.4 : Evaluate models: metrics, size sensitivity, t-SNE visualization

> **Input:** `clean_football_kb.ttl` (64,910 triples, 38,655 entities, 459 relations)  
> **Output:** `augmented_football_kb.ttl` (+1,294 inferred triples) + trained KGE models

## C.1 : Preparation of KGE Datasets (Train / Valid / Test)

Before training any KGE model, the raw Knowledge Graph must be prepared into three splits. This step involves sevrals operations.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ← Ton dossier Drive mis à jour
WORK_DIR = "/content/drive/MyDrive/Part C/Project Web Datamining & Semantics"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"current directory: {os.getcwd()}")
print(f"files present: {os.listdir(WORK_DIR)}")

### Step 1 : Filtering

KGE models strictly require relational triples (head, relation, tail) where all elements are URIs (Unique Resource Identifiers). We must filter out Literals (strings, numbers) and Blank Nodes (anonymous RDF nodes).

Only structural relations are kept.   
example :`ex:Mohamed_Salah → prop:playsFor → ex:Liverpool_FC`.

In [ ]:
!pip install rdflib

In [ ]:
import os
import random
from rdflib import Graph, URIRef

In [ ]:
graph_file = "clean_football_kb.ttl"
g = Graph()
g.parse(graph_file, format="turtle")
print(f"numb of triplets: {len(g)}")

In [ ]:
triplets = []
for s, p, o in g:
    if isinstance(s, URIRef) and isinstance(p, URIRef) and isinstance(o, URIRef):
        triplets.append((str(s), str(p), str(o)))

print(f"Total relational triples retained for KGE: {len(triplets)}")

### Step 2 : Shuffling & Splitting (80/10/10)

We randomly shuffle the triples and split them into three sets:
   - Training set (80%): Used by the model to learn the entity and relation embeddings.
   - Validation set (10%): Used to tune hyperparameters and prevent overfitting.
   - Testing set (10%): Held out until the end to evaluate the model's true performance.

In [ ]:
random.seed(42)
random.shuffle(triplets)

In [ ]:
total_triplets = len(triplets)
train_idx = int(total_triplets * 0.8)
valid_idx = int(total_triplets * 0.9)

train_triplets = triplets[:train_idx]
valid_triplets = triplets[train_idx:valid_idx]
test_triplets = triplets[valid_idx:]

print(f" - Train : {len(train_triplets)} triplets")
print(f" - Valid : {len(valid_triplets)} triplets")
print(f" - Test  : {len(test_triplets)} triplets")

### STEP 3: Preventing Out-Of-Vocabulary Entities (OOV)

 A KGE model cannot evaluate an entity or relation it has never seen during training. After the initial split, we systematically check the Validation and Test sets. Any triple containing an entity or relation missing from the Training set is moved into the Training set. This guarantees 100% graph connectivity during evaluation.

In [ ]:
def get_vocab(triplet_list):
    entities, relations = set(), set()
    for s, p, o in triplet_list:
        entities.add(s)
        entities.add(o)
        relations.add(p)
    return entities, relations

In [ ]:
train_entities, train_relations = get_vocab(train_triplets)

In [ ]:
def fix_oov(target_triplets, train_triplets, train_entities, train_relations):
    fixed_target = []
    moved_count = 0
    for s, p, o in target_triplets:
        if s not in train_entities or o not in train_entities or p not in train_relations:
            train_triplets.append((s, p, o))
            train_entities.add(s)
            train_entities.add(o)
            train_relations.add(p)
            moved_count += 1
        else:
            fixed_target.append((s, p, o))
    return fixed_target, moved_count

In [ ]:
valid_triplets, moved_val = fix_oov(valid_triplets, train_triplets, train_entities, train_relations)

test_triplets, moved_test = fix_oov(test_triplets, train_triplets, train_entities, train_relations)

In [ ]:
print(f"\nOOV Prevention Fixes:")
print(f" - Moved {moved_val} triples from Valid to Train.")
print(f" - Moved {moved_test} triples from Test to Train.")

print(f"\nFinal Data split breakdown:")
print(f" - Train : {len(train_triplets)} triples")
print(f" - Valid : {len(valid_triplets)} triples")
print(f" - Test  : {len(test_triplets)} triples")

### Step 4 : Exporting:  
 We save the splits into ".txt" files in a Tab-Separated Values "\t" format, which is the standard input format expected by KGE libraries like PyKEEN.

In [ ]:
def format_tsv(triplet_list):
    return [f"{s}\t{p}\t{o}\n" for s, p, o in triplet_list]

In [ ]:
dataset_dir = "kge_dataset"
os.makedirs(dataset_dir, exist_ok=True)

with open(os.path.join(dataset_dir, "train.txt"), "w", encoding="utf-8") as f:
    f.writelines(format_tsv(train_triplets))

with open(os.path.join(dataset_dir, "valid.txt"), "w", encoding="utf-8") as f:
    f.writelines(format_tsv(valid_triplets))

with open(os.path.join(dataset_dir, "test.txt"), "w", encoding="utf-8") as f:
    f.writelines(format_tsv(test_triplets))

print(f"\ntrain.txt, valid.txt, and test.txt in the '{dataset_dir}/' directory.")

### C.1 Results

| Split | Triples | % |
|---|---|---|
| Train | 58,491 | ~80% |
| Valid | 3,210 | ~10% |
| Test | 3,209 | ~10% |
| **Total** | **64,910** | **100%** |

The graph contains **38,655 unique entities** and **459 distinct relations**. The large number of relations (459) comes from the Wikidata alignment in Part B, which imported diverse Wikidata predicates beyond the 7 core football properties. OOV prevention moved a small number of triples to train, ensuring complete vocabulary coverage across all three splits.

## C.2 : Symbolic Reasoning with SWRL and Owlready2

Knowledge Graphs allow explicitly defining rules to infer new, hidden knowledge not directly stated in the dataset. We declare two SWRL (Semantic Web Rule Language) rules using "owlready2".

| Rule | Logic | New Property |
|---|---|---|
| coachedBy | `playsFor(?p,?t) ∧ headCoach(?t,?c) → coachedBy(?p,?c)` | Person → Person |
| competesIn | `playsFor(?p,?t) ∧ locatedIn(?t,?c) → competesIn(?p,?c)` | Person → Country |

In [ ]:
!pip install owlready2

Imports

In [ ]:
import rdflib
from rdflib import RDF, OWL, Namespace
from rdflib.namespace import RDFS
from owlready2 import World, Thing, ObjectProperty, Imp
import os

### 1. Grpah Preparation (RDFLIB)

We load "clean_football_kb.ttl" and enrich it with OWL declarations (classes, object properties, domain/range constraints) so owlready2 can understand the ontology. All instances are typed as "OWL.NamedIndividual", which is required by owlready2 for SWRL rule matching. The graph is serialized to XML/OWL format as an intermediate step.

In [ ]:
g = rdflib.Graph()
g.parse("clean_football_kb.ttl", format="turtle")


ONTO_IRI = "http://www.example.org/football"
EX       = Namespace("http://www.example.org/football/")       # entités
EX_PROP  = Namespace("http://www.example.org/football/prop/")  # propriétés


g.add((URIRef(ONTO_IRI), RDF.type, OWL.Ontology))


for cls in ["Person", "Team", "Country", "FootballPosition"]:
    g.add((EX[cls], RDF.type, OWL.Class))

# Declaration of properties (existing + inferred)
for p in ["playsFor", "headCoach", "locatedIn",
          "nationality", "playsPosition", "coachedBy", "competesIn"]:
    g.add((EX_PROP[p], RDF.type, OWL.ObjectProperty))


g.add((EX_PROP["coachedBy"],  RDFS.domain, EX["Person"]))
g.add((EX_PROP["coachedBy"],  RDFS.range,  EX["Person"]))
g.add((EX_PROP["competesIn"], RDFS.domain, EX["Person"]))
g.add((EX_PROP["competesIn"], RDFS.range,  EX["Country"]))


IGNORE = {OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty,
          OWL.AnnotationProperty, OWL.Ontology, OWL.NamedIndividual}
for s, p, o in list(g.triples((None, RDF.type, None))):
    if o not in IGNORE and isinstance(s, URIRef):
        g.add((s, RDF.type, OWL.NamedIndividual))

count_ni = len(list(g.triples((None, RDF.type, OWL.NamedIndividual))))
print(f"   -> {count_ni} entities NamedIndividual")


for prop in ["playsFor", "locatedIn", "headCoach"]:
    triples = list(g.triples((None, EX_PROP[prop], None)))
    ex = triples[0] if triples else None
    print(f"   -> '{prop}' : {len(triples)} triplets" +
          (f"  (ex: {ex[0].split('/')[-1]} → {ex[2].split('/')[-1]})" if ex else "Nothing"))


xml_file = os.path.abspath("clean_football_kb_typed.xml")
g.serialize(xml_file, format="xml")
print(f"   -> XML file: {xml_file}")

### 2. Owlready2 schema

The XML ontology is loaded into a "World" object. Two namespaces are used: the entity namespace (ex:) for classes like Person and Team, and the property namespace (prop:) for relations like playsFor and headCoach. Both must be explicitly passed to set_as_rule() so owlready2 can resolve the entity names.

In [ ]:
my_world = World()
onto     = my_world.get_ontology(f"file://{xml_file}").load()

Person  = onto.search_one(iri=str(EX["Person"]))
Team    = onto.search_one(iri=str(EX["Team"]))
Country = onto.search_one(iri=str(EX["Country"]))

if not all([Person, Team, Country]):
    print("Classes not found :")
    for i, e in enumerate(list(onto.individuals())[:5]):
        print(f"     {e.iri}")
    raise RuntimeError("Check EX and EX_PROP namespaces")

print(f"   -> Person={Person.name} | Team={Team.name} | Country={Country.name}")

prop_ns = my_world.get_namespace("http://www.example.org/football/prop/")

### 3. SWRL rules declared

Two `Imp()` objects are created and registered in the ontology. These rules shows that a player inherits the coach of their team `coachedBy`, and a player competes in the country where their team is located `competesIn`. These facts cannot be read directly from the source data so they must be inferred.

In [ ]:
with onto:
    # Rules 1
    rule1 = Imp()
    rule1.set_as_rule(
        "Person(?p), Team(?t), Person(?c), "
        "playsFor(?p, ?t), headCoach(?t, ?c) -> coachedBy(?p, ?c)",
        namespaces=[onto, prop_ns]
    )
    # Rules 2
    rule2 = Imp()
    rule2.set_as_rule(
        "Person(?p), Team(?t), Country(?c), "
        "playsFor(?p, ?t), locatedIn(?t, ?c) -> competesIn(?p, ?c)",
        namespaces=[onto, prop_ns]
    )

print("-> Rules 1 : Person(?p) ∧ Team(?t) ∧ Person(?c)  ∧ playsFor(?p,?t) ∧ headCoach(?t,?c) → coachedBy(?p,?c)")
print("-> Rules 2 : Person(?p) ∧ Team(?t) ∧ Country(?c) ∧ playsFor(?p,?t) ∧ locatedIn(?t,?c) → competesIn(?p,?c)")

### 4.Rules applied via SPARQL CONSTRUCT

Each rule is translated into a SPARQL CONSTRUCT query that matches the antecedent and generates the consequent triples. Both rules produce **647 new triples each**, one per player in the dataset (all 20 Premier League teams × their full squads), because every player plays for exactly one team, so every player gets exactly 1 coach and 1 "compete in" link.

In [ ]:
g.bind("ex",   EX)
g.bind("prop", EX_PROP)
g.bind("owl",  OWL)

SPARQL_RULE1 = """
PREFIX ex:   <http://www.example.org/football/>
PREFIX prop: <http://www.example.org/football/prop/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
CONSTRUCT { ?p prop:coachedBy ?c . }
WHERE {
    ?p  rdf:type       ex:Person .
    ?t  rdf:type       ex:Team   .
    ?c  rdf:type       ex:Person .
    ?p  prop:playsFor  ?t .
    ?t  prop:headCoach ?c .
}
"""

SPARQL_RULE2 = """
PREFIX ex:   <http://www.example.org/football/>
PREFIX prop: <http://www.example.org/football/prop/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
CONSTRUCT { ?p prop:competesIn ?c . }
WHERE {
    ?p  rdf:type        ex:Person  .
    ?t  rdf:type        ex:Team    .
    ?c  rdf:type        ex:Country .
    ?p  prop:playsFor   ?t .
    ?t  prop:locatedIn  ?c .
}
"""

new_triples_1 = list(g.query(SPARQL_RULE1))
for triple in new_triples_1:
    g.add(triple)
count_coached = len(new_triples_1)

new_triples_2 = list(g.query(SPARQL_RULE2))
for triple in new_triples_2:
    g.add(triple)
count_competes = len(new_triples_2)

print(f"Rules 1 (coachedBy)  -> {count_coached}  new triplets")
print(f"Rules 2 (competesIn) -> {count_competes} new triplets")

### 5. Check of inferred knowledge

In [ ]:
print("\nExample : competesIn")
q_ex_competes = """
PREFIX prop: <http://www.example.org/football/prop/>
SELECT ?player ?country WHERE { ?player prop:competesIn ?country . } LIMIT 5
"""
for row in g.query(q_ex_competes):
    player  = str(row.player).split('/')[-1]
    country = str(row.country).split('/')[-1]
    print(f"     {player} compète en {country}")

print("\nExample : coachedBy")
q_ex_coached = """
PREFIX prop: <http://www.example.org/football/prop/>
SELECT ?player ?coach WHERE { ?player prop:coachedBy ?coach . } LIMIT 5
"""
for row in g.query(q_ex_coached):
    player = str(row.player).split('/')[-1]
    coach  = str(row.coach).split('/')[-1]
    print(f"     {player} est coaché par {coach}")

print("\nTop 5 players by competition country")
q_by_country = """
PREFIX prop: <http://www.example.org/football/prop/>
SELECT ?country (COUNT(?player) AS ?nb)
WHERE { ?player prop:competesIn ?country . }
GROUP BY ?country ORDER BY DESC(?nb) LIMIT 5
"""
for row in g.query(q_by_country):
    country = str(row.country).split('/')[-1]
    print(f"     {country} : {int(row.nb)} joueurs")

 ### 6. Grpah backup

In [ ]:
augmented_ttl = "augmented_football_kb.ttl"
augmented_xml = "augmented_football_kb.xml"

g.serialize(augmented_ttl, format="turtle")
onto.save(file=augmented_xml, format="rdfxml")

total_triples = len(list(g.triples((None, None, None))))

print(f"Total final triples:, {total_triples}")

### C.2 Results

| Metric | Value |
|---|---|
| Initial triples | 65,702 |
| New `coachedBy` triples | **647** |
| New `competesIn` triples | **647** |
| **Total triples (augmented)** | **66,996** |

These two inferred properties are critical for the RAG pipeline they can answer questions like *Which players compete in England?* or *Who coaches Mohamed Salah?* correctly.

## C.3 : KGE Model Training: TransE and ComplEx

We train two KGE models using PyKEEN, Both models learn dense vector representations (embeddings) of entities and relations such that the graph structure is preserved in the embedding space.

Import

In [ ]:
!pip install pykeen torch --quiet

In [ ]:
import torch
import pykeen
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
from pykeen.models import TransE, ComplEx
import pandas as pd
import matplotlib.pyplot as plt
import os, json, time

### 1. Train, valid, test data loaded



In [ ]:
TRAIN_PATH = "kge_dataset/train.txt"
VALID_PATH  = "kge_dataset/valid.txt"
TEST_PATH   = "kge_dataset/test.txt"

tf_train = TriplesFactory.from_path(
    TRAIN_PATH,
    delimiter="\t"
)
tf_valid = TriplesFactory.from_path(
    VALID_PATH,
    delimiter="\t",
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id,
)
tf_test = TriplesFactory.from_path(
    TEST_PATH,
    delimiter="\t",
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id,
)

print(f" Train : {tf_train.num_triples:>6,} triplets")
print(f" Valid : {tf_valid.num_triples:>6,} triplets")
print(f" Test : {tf_test.num_triples:>6,} triplets")
print(f" Entités : {tf_train.num_entities:,}")
print(f" Relations : {tf_train.num_relations:,}")

### 2. Models configuration

Hyperparameters: `embedding_dim=128`, `epochs=200`, `batch_size=256`, `lr=0.001` (Adam)

In [ ]:
EMBEDDING_DIM = 128
EPOCHS        = 200
BATCH_SIZE    = 256
LR            = 0.001

MODEL_CONFIGS = {
    "TransE": {
        "model": "TransE",
        "model_kwargs": {
            "embedding_dim": EMBEDDING_DIM,
        },
        "loss":    "MarginRankingLoss",
        "color":   "#E74C3C",
    },
    "ComplEx": {
        "model": "ComplEx",
        "model_kwargs": {
            "embedding_dim": EMBEDDING_DIM,
        },
        "loss":    "SoftplusLoss",
        "color":   "#2980B9",
    },
}

for name, cfg in MODEL_CONFIGS.items():
    print(f"   {name:10s} | dim={EMBEDDING_DIM} | loss={cfg['loss']}")

### 3. Training

In [ ]:
results_store = {}

for model_name, cfg in MODEL_CONFIGS.items():
    print(f"\n{'─'*55}")
    print(f"  Training : {model_name}")
    print(f"{'─'*55}")
    t_start = time.time()

    result = pipeline(

        training=tf_train,
        validation=tf_valid,
        testing=tf_test,

        model=cfg["model"],
        model_kwargs=cfg["model_kwargs"],

        optimizer="Adam",
        optimizer_kwargs={"lr": LR},
        loss=cfg["loss"],

        training_kwargs={
            "num_epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
        },

        evaluation_kwargs={"batch_size": BATCH_SIZE},

        random_seed=42,
        device=device,
    )

    elapsed = time.time() - t_start
    results_store[model_name] = {
        "pipeline_result": result,
        "elapsed":         elapsed,
    }

Summary

In [ ]:
metrics = result.metric_results.to_flat_dict()
    mrr = metrics.get("both.realistic.inverse_harmonic_mean_rank", 0)
    hits1 = metrics.get("both.realistic.hits_at_1", 0)
    hits10  = metrics.get("both.realistic.hits_at_10", 0)
    print(f" {model_name} entraîné en {elapsed:.0f}s")
    print(f" MRR : {mrr:.4f}")
    print(f" Hits@1 : {hits1:.4f}")
    print(f" Hits@10 : {hits10:.4f}")

    save_dir = f"models/kge_{model_name.lower()}"
    os.makedirs(save_dir, exist_ok=True)
    result.save_to_directory(save_dir)

TransE trained in 318s, ComplEx in 604s. ComplEx takes longer because it operates in complex number space, effectively doubling the number of parameters compared to TransE for the same embedding dimension.

In [ ]:
result_transe = results_store["TransE"]["pipeline_result"]
all_metrics = result_transe.metric_results.to_flat_dict()

for key, val in sorted(all_metrics.items()):
    print(f"  {key:<60} : {val:.4f}")

### 4. Table of metrics

In [ ]:
METRIC_KEYS = {
    "MRR"     : "both.realistic.inverse_harmonic_mean_rank",
    "Hits@1"  : "both.realistic.hits_at_1",
    "Hits@3"  : "both.realistic.hits_at_3",
    "Hits@10" : "both.realistic.hits_at_10",
    "MR"      : "both.realistic.mean_rank",
}

rows = []
for model_name, data in results_store.items():
    metrics = data["pipeline_result"].metric_results.to_flat_dict()
    row = {"Modele": model_name}
    for label, key in METRIC_KEYS.items():
        val = metrics.get(key, float("nan"))
        row[label] = round(float(val), 4)
    row["Time (s)"] = round(data["elapsed"], 1)
    rows.append(row)

df_metrics = pd.DataFrame(rows).set_index("Modele")
print(df_metrics.to_string())

df_metrics.to_csv("data/kge_metrics_comparison.csv")

### C.3 Results — Comparative Metrics

| Model | MRR | Hits@1 | Hits@3 | Hits@10 | Training Time |
|---|---|---|---|---|---|
| **TransE** | **0.1601** | **0.0842** | **0.1783** | **0.3093** | 318s |
| ComplEx | 0.0244 | 0.0099 | 0.0206 | 0.0498 | 604s |

TransE significantly outperforms ComplEx. The reason is because the graph is very sparse (1.5 triples/entity average) and TransE is more robust. Also, the relations are asymmetric and dominated by "playsFor", so TransE handles this well. I, addition, ComplEx's bilinear scoring benefits more from denser graphs with diverse relation types.

### 5. Visualization : Loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("C.3 : KGE Training Loss : TransE vs ComplEx", fontsize=14, fontweight="bold")

for ax, (model_name, data) in zip(axes, results_store.items()):
    losses = data["pipeline_result"].losses
    color  = MODEL_CONFIGS[model_name]["color"]
    ax.plot(losses, color=color, linewidth=1.5, label=model_name)
    ax.set_title(model_name, fontsize=12)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax.annotate(
        f"Final : {losses[-1]:.4f}",
        xy=(len(losses)-1, losses[-1]),
        xytext=(-60, 20), textcoords="offset points",
        arrowprops=dict(arrowstyle="->", color="gray"),
        fontsize=9, color=color
    )

plt.tight_layout()
plt.savefig("graphs/kge_training_loss.png", dpi=150, bbox_inches="tight")
plt.show()

Both models converge over 200 epochs. For the **TransE**, it starts at 0.9, converges to **0.0031**. There is a rapid initial learning, then decrease.
For the **ComplEx**, it starts at 9.0, converges to **0.5567**. The higher final loss is expected because SoftplusLoss operates on a different scale than MarginRankingLoss

The convergence of both curves confirms that the hyperparameters (lr=0.001, batch_size=256) are well for this dataset.

## C.4 : In-Depth Evaluation, Size Sensitivity and Visualization

This section offers a comprehensive evaluation of the trained KGE models by examining performance. We begin with a prediction metrics, specifically comparing head versus tail entity performance to identify potential directional biases. To understand how the models scale, a sensitivity analysis shows how performance across different training data volumes, ranging from 20k samples to the full dataset. Furthermore, we employ t-SNE to project the 128D embeddings into a 2D space, visually uncovering hidden semantic clusters, which are then validated through a nearest neighbor analysis to ensure that similar entities are positioned within the embedding space.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
import warnings, time, os
warnings.filterwarnings("ignore")

### 1. Head vs Tail Prediction Analysis

In [ ]:
DIRECTIONS = ["head", "tail", "both"]
METRICS_LABELS = {
    "MRR"     : "inverse_harmonic_mean_rank",
    "Hits@1"  : "hits_at_1",
    "Hits@3"  : "hits_at_3",
    "Hits@10" : "hits_at_10",
    "MR"      : "arithmetic_mean_rank",
}

rows_full = []
for model_name, data in results_store.items():
    flat = data["pipeline_result"].metric_results.to_flat_dict()
    for direction in DIRECTIONS:
        row = {"Modèle": model_name, "Direction": direction}
        for label, key_suffix in METRICS_LABELS.items():
            full_key = f"{direction}.realistic.{key_suffix}"
            val = flat.get(full_key, float("nan"))
            row[label] = round(float(val), 4)
        rows_full.append(row)

df_full = pd.DataFrame(rows_full).set_index(["Modele", "Direction"])
print(df_full.to_string())
df_full.to_csv("data/kge_metrics_full.csv")

| Model | Direction | MRR | Hits@1 | Hits@10 | MR |
|---|---|---|---|---|---|
| TransE | head | 0.1132 | 0.0440 | 0.2501 | 906 |
| TransE | **tail** | **0.1975** | **0.1117** | **0.3685** | 1,054 |
| ComplEx | head | 0.0142 | 0.0028 | 0.0289 | 9,873 |
| ComplEx | tail | 0.0319 | 0.0157 | 0.0689 | 10,205 |

TransE predicts tails better than heads (MRR: 0.1975 vs 0.1132). This is expected because tail prediction for "playsFor" means predicting which team a player plays for, and there are only 20 possible teams, making it much easier than head prediction (predicting which of 667 players is associated with a given team/relation). ComplEx's very high Mean Rank (10,000) confirms it struggles on this sparse graph.

### 2. Size Sensitivity Results

In [ ]:
SIZES = {
    "20k" : 20_000,
    "50k" : 50_000,
    "full": tf_train.num_triples,  # ~58k
}

sensitivity_results = {}  # {model: {size_label: metrics_dict}}

for model_name, cfg in MODEL_CONFIGS.items():
    sensitivity_results[model_name] = {}

    flat_full = results_store[model_name]["pipeline_result"].metric_results.to_flat_dict()
    sensitivity_results[model_name]["full"] = flat_full

    for size_label, n_triples in [("20k", 20_000), ("50k", 50_000)]:
        print(f"   [{model_name}] Taille : {size_label} ({n_triples:,} triplets)...")

        np.random.seed(42)
        idx = np.random.choice(tf_train.num_triples, size=n_triples, replace=False)
        mapped = tf_train.mapped_triples[idx]

        tf_sub = TriplesFactory(
            mapped_triples=mapped,
            entity_to_id=tf_train.entity_to_id,
            relation_to_id=tf_train.relation_to_id,
        )

        t0 = time.time()
        result_sub = pipeline(
            training=tf_sub,
            validation=tf_valid,
            testing=tf_test,
            model=cfg["model"],
            model_kwargs=cfg["model_kwargs"],
            optimizer="Adam",
            optimizer_kwargs={"lr": LR},
            loss=cfg["loss"],
            training_kwargs={"num_epochs": EPOCHS, "batch_size": BATCH_SIZE},
            evaluation_kwargs={"batch_size": BATCH_SIZE},
            random_seed=42,
            device=device,
        )
        elapsed = time.time() - t0

        flat = result_sub.metric_results.to_flat_dict()
        sensitivity_results[model_name][size_label] = flat

        mrr  = flat.get("both.realistic.inverse_harmonic_mean_rank", 0)
        h10  = flat.get("both.realistic.hits_at_10", 0)
        print(f"      ✓ MRR={mrr:.4f} | H@10={h10:.4f} | {elapsed:.0f}s")

| Model | 20k MRR | 50k MRR | full MRR | Trend |
|---|---|---|---|---|
| TransE | 0.0678 | 0.1366 | **0.1601** | Steady linear growth |
| ComplEx | 0.0003 | 0.0095 | **0.0244** | Exponential — needs critical mass |

TransE shows consistent and improves with data volume. Hit performance doubles as training data doubles. This makes it reliable even with limited data.

ComplEx is almost non functional at 20k (MRR=0.0003) and only starts learning meaningfully at 50k. This shows that ComplEx requires a mass of training examples to capture the patterns. With the full dataset (58k), it still lags far behind TransE.

Backup

In [ ]:
import json

sensitivity_to_save = {}
for model_name, size_dict in sensitivity_results.items():
    sensitivity_to_save[model_name] = {}
    for size_label, flat in size_dict.items():
        sensitivity_to_save[model_name][size_label] = {
            k: float(v) for k, v in flat.items()
        }

with open("data/sensitivity_results.json", "w") as f:
    json.dump(sensitivity_to_save, f, indent=2)

### 3. Sensibility visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("KGE Sensibility", fontsize=14, fontweight="bold")

SIZE_ORDER  = ["20k", "50k", "full"]
SIZE_VALUES = [20_000, 50_000, tf_train.num_triples]

for ax, (model_name, cfg) in zip(axes, MODEL_CONFIGS.items()):
    color = cfg["color"]
    size_results = sensitivity_results[model_name]

    mrr_vals  = [size_results[s].get("both.realistic.inverse_harmonic_mean_rank", 0) for s in SIZE_ORDER]
    h1_vals   = [size_results[s].get("both.realistic.hits_at_1", 0)   for s in SIZE_ORDER]
    h10_vals  = [size_results[s].get("both.realistic.hits_at_10", 0)  for s in SIZE_ORDER]

    x = SIZE_VALUES
    ax.plot(x, mrr_vals,  "o-",  color=color,   linewidth=2, label="MRR",     markersize=8)
    ax.plot(x, h1_vals,   "s--", color=color,   linewidth=1.5, label="Hits@1", markersize=7, alpha=0.7)
    ax.plot(x, h10_vals,  "^-.", color=color,   linewidth=1.5, label="Hits@10",markersize=7, alpha=0.7)

    ax.set_title(model_name, fontsize=12)
    ax.set_xlabel("training set size (triplets)")
    ax.set_ylabel("Score")
    ax.set_xticks(SIZE_VALUES)
    ax.set_xticklabels(SIZE_ORDER)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(max(h10_vals) * 1.2, 0.05))

plt.tight_layout()
plt.savefig("graphs/kge_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

The sensitivity curves confirm the trends above. TransE's three metrics (MRR, Hits@1, Hits@10) all grow in parallel with data size, suggesting the model architecture is a good fit for this graph. ComplEx's curves remain near-zero until the full dataset, indicating it would benefit significantly from either more data or dedicated hyperparameter tuning (lower learning rate, L2 regularization).

### 4. t-SNE Visualization

In [ ]:
transe_model = results_store["TransE"]["pipeline_result"].model
entity_embeddings = transe_model.entity_representations[0](
    indices=None
).detach().cpu().numpy()

print(f"-> Embeddings: {entity_embeddings.shape}  (entités × dim)")

# Récupération du mapping id → nom d'entité
id_to_entity = {v: k for k, v in tf_train.entity_to_id.items()}

# Classification des entités par type (depuis le nom de l'URI)
EX_BASE = "http://www.example.org/football/"

def get_entity_type(uri):
    name = uri.replace(EX_BASE, "")
    # Heuristiques simples basées sur la structure du KG
    if any(suffix in name for suffix in ["_FC", "_AFC", "_United", "_City",
                                          "_Hotspur", "_Athletic", "_Wanderers",
                                          "Bournemouth", "Brighton", "Brentford",
                                          "Chelsea", "Arsenal", "Liverpool",
                                          "Palace", "Everton", "Fulham",
                                          "Nottingham", "Sunderland", "Ipswich",
                                          "Leicester", "Southampton", "Leeds"]):
        return "Team"
    if any(c.islower() for c in name) and "_" in name and len(name) > 6:
        return "Person"
    if name[0].isupper() and "_" not in name:
        return "Country/Position"
    return "Other"


n_entities = entity_embeddings.shape[0]
entity_types = [get_entity_type(id_to_entity.get(i, "")) for i in range(n_entities)]

type_to_color = {
    "Person"           : "#3498DB",
    "Team"             : "#E74C3C",
    "Country/Position" : "#2ECC71",
    "Other"            : "#95A5A6",
}

N_SAMPLE = min(3000, n_entities)
np.random.seed(42)
sample_idx = np.random.choice(n_entities, size=N_SAMPLE, replace=False)

emb_sample   = entity_embeddings[sample_idx]
types_sample = [entity_types[i] for i in sample_idx]

print(f"   -> t-SNE {N_SAMPLE} entities (dim 128 → 2)...")
tsne = TSNE(
    n_components=2,
    perplexity=40,
    n_iter=1000,
    random_state=42,
    init="pca",
    learning_rate="auto",
)
emb_2d = tsne.fit_transform(emb_sample)
print("   -> t-SNE finish")

# Plot
fig, ax = plt.subplots(figsize=(13, 9))
fig.suptitle("t-SNE of TransE embeddings (dim 128 → 2D)", fontsize=14, fontweight="bold")

for etype, color in type_to_color.items():
    mask = [i for i, t in enumerate(types_sample) if t == etype]
    if mask:
        ax.scatter(
            emb_2d[mask, 0], emb_2d[mask, 1],
            c=color, label=etype,
            s=12, alpha=0.6, linewidths=0
        )

team_indices = [i for i, t in enumerate(types_sample) if t == "Team"]
for idx in team_indices[:15]:
    entity_uri = id_to_entity.get(sample_idx[idx], "")
    name = entity_uri.replace(EX_BASE, "").replace("_FC", "").replace("_", " ")
    ax.annotate(name, (emb_2d[idx, 0], emb_2d[idx, 1]),
                fontsize=7, alpha=0.85,
                xytext=(4, 4), textcoords="offset points")

ax.legend(fontsize=11, markerscale=2)
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("graphs/kge_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

t-SNE projects the 128-dimensional TransE entity embeddings into 2D while preserving local neighborhood structure. Entities are color-coded by type: in blue Person (players & coaches), in red Team (Premier League clubs) and in green
Country/Position.

A well-trained model should produce visible clusters where entities of the same type and especially entities sharing many relations group together in the 2D space.

### 5. Nearest Neighbor Validation

In [ ]:
def nearest_neighbors(entity_uri, embeddings, entity_to_id, id_to_entity, top_k=5):
    if entity_uri not in entity_to_id:
        return []
    idx    = entity_to_id[entity_uri]
    vec    = embeddings[idx].reshape(1, -1)
    sims   = cosine_similarity(vec, embeddings)[0]
    sims[idx] = -1
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(id_to_entity[i], round(float(sims[i]), 4)) for i in top_idx]


PROBE_ENTITIES = [
    f"{EX_BASE}Arsenal_FC",
    f"{EX_BASE}Chelsea_FC",
    f"{EX_BASE}England",
    f"{EX_BASE}Mohamed_Salah",
]

print()
for probe_uri in PROBE_ENTITIES:
    name = probe_uri.replace(EX_BASE, "").replace("_", " ")
    neighbors = nearest_neighbors(
        probe_uri, entity_embeddings,
        tf_train.entity_to_id, id_to_entity, top_k=5
    )
    if neighbors:
        print(f" Neighbor of '{name}':")
        for neighbor_uri, sim in neighbors:
            neighbor_name = neighbor_uri.replace(EX_BASE, "").replace("_", " ")
            print(f"      {neighbor_name:<35} (sim={sim:.4f})")
    else:
        print(f"'{name}'not found.")
    print()



Nearest neighbors confirm that TransE has learned semantically meaningful representations:

| Query Entity | Top Neighbors | Interpretation |
|---|---|---|
| Arsenal FC | Leeds Utd, Man City, Brighton, Burnley, Spurs | ✅ All Premier League clubs |
| England | Sweden, Wales, Norway, Serbia, USA | ✅ European/international nations |
| Mohamed Salah | Ismaïla Sarr, Lorenzo Lucca, James Milner | ✅ International attackers |

A raw Wikidata URI appears in Chelsea's neighbors, a trace of the Part B alignment where some entities were not fully resolved to local URIs. ALso,
a Chelsea player (Josh Acheampong) appears in Chelsea FC's neighbors because the model correctly places players close to their teams. This is a correct behavior, not a bug.

### C.4 Summary

| Section | Key Result |
|---|---|
| C.1 | 58,491 train / 3,210 valid / 3,209 test — 38,655 entities, 459 relations |
| C.2 | 2 SWRL rules → +647 `competesIn` + 647 `coachedBy` inferred triples |
| C.3 | TransE (MRR=0.1601) >> ComplEx (MRR=0.0244) — both converge in <200 epochs |
| C.4 | Tail prediction > head; TransE scales linearly; t-SNE shows semantic clusters |

The trained TransE model and augmented KG `augmented_football_kb.ttl` are now available.